# Mission 3: 고정 0.5 / 분류 head 비교
새 Colab GPU 노트북에서 위에서 아래로 실행하세요. 기존 노트북은 보관합니다.

우선순위: **기존 오류 분석 → CLS 대조군 → Mean pooling → 증상별 Attention**.
모든 실험은 Base / First-512 / BCE / 동일 분할 / seed 42 / 4 epoch / 마지막 epoch 평가로 통일합니다.
새 head 효과만 비교하기 위해 이번 파일에서는 손실함수·입력 길이·임계값을 바꾸지 않습니다.
CLS 대조군은 같은 구현·환경의 기준이며 기존 E0 0.6133의 정확한 재현을 보장하지 않습니다.
결과를 본 뒤 선택하므로 내부 검증은 탐색용이며 독립 테스트 점수가 아닙니다.

준비 파일: 원본 Training ZIP(Drive), split_manifest.csv, m3_fixed_results.tar.gz.
.ipynb는 모델/예측 파일을 포함하지 않습니다. 마지막 백업 셀을 꼭 실행하세요.
**전체 실행 시 세 모델이 순차 학습됩니다. 한 단계씩 실행하는 것을 권합니다.**


In [ ]:
%pip install -q "transformers>=4.46,<6" "accelerate>=1,<2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" safetensors


## 1. 경로·환경
GPU를 선택하세요. 다운로드한 작은 결과 백업과 manifest는 Colab 왼쪽 파일 패널로 /content에 업로드합니다.
원본 모델 압축파일은 필요 없습니다. Drive 경로는 실제 위치에 맞게 수정하세요.


In [ ]:
from pathlib import Path
import json, zipfile, tarfile, hashlib, gc, inspect, platform
import importlib.metadata as im
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, RobertaModel, RobertaPreTrainedModel,
                          TrainingArguments, Trainer, set_seed, RobertaConfig)
from transformers.modeling_outputs import SequenceClassifierOutput
from google.colab import drive, files
drive.mount("/content/drive")

TRAIN_ZIP = Path("/content/drive/MyDrive/DDC_M3/data/mission3_train_json.zip")
MANIFEST = Path("/content/split_manifest.csv")
OLD_BACKUP = Path("/content/m3_fixed_results.tar.gz")
OUT = Path("/content/m3_head_comparison_v1")
OUT.mkdir(exist_ok=True)
TARGETS = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]
MODEL_NAME = "klue/roberta-base"
assert torch.cuda.is_available(), "먼저 GPU 런타임을 선택하세요."
ENV = {"python": platform.python_version(), **{n: im.version(n) for n in
       ["torch","transformers","accelerate","numpy","pandas","scikit-learn"]}}
print(ENV)
print("GPU:", torch.cuda.get_device_name(0))
for p in [TRAIN_ZIP, MANIFEST, OLD_BACKUP]:
    assert p.is_file(), f"파일을 준비하세요: {p}"


## 2. 원본 분할·텍스트 복원
Training만 읽습니다. 공식 Validation은 사용하지 않습니다.
파일 ID는 분할·예측 정렬 검사용이며 모델 입력에 포함하지 않습니다.


In [ ]:
rows = []
with zipfile.ZipFile(TRAIN_ZIP) as z:
    for name in sorted(z.namelist()):
        p = Path(name)
        if "__MACOSX" in p.parts or p.name.startswith("._") or p.suffix.lower() != ".json":
            continue
        d = json.loads(z.read(name))
        texts = [u.get("text", "") for u in d["utterances"]]
        assert all(isinstance(t, str) for t in texts)
        assert isinstance(d["symptom"], list)
        rows.append({"file_name": p.name, "text": " ".join(texts),
                     **{t: int(t in d["symptom"]) for t in TARGETS}})
df = pd.DataFrame(rows)
split = pd.read_csv(MANIFEST)
assert len(df) == 29200 and not df.file_name.duplicated().any()
assert not split.file_name.duplicated().any()
assert set(split.file_name) == set(df.file_name)
assert set(split.split) == {"train","dev_valid"}
ordered = split.merge(df, on="file_name", how="left", validate="one_to_one", sort=False)
train = ordered[ordered.split == "train"].reset_index(drop=True)
valid = ordered[ordered.split == "dev_valid"].reset_index(drop=True)
assert (len(train), len(valid)) == (23408, 5792)
split.to_csv(OUT / "split_manifest.csv", index=False)
print("내부 학습:", train.shape, "내부 검증:", valid.shape)
print("manifest SHA256:", hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

# 압축 해제 없이 필요한 예측만 읽습니다.
import io
old = {}
with tarfile.open(OLD_BACKUP, "r:gz") as archive:
    for run in ["tfidf","e0","e1","e3","e0_e3_equal"]:
        member = archive.getmember(f"{run}/dev_predictions.npz")
        with np.load(io.BytesIO(archive.extractfile(member).read()), allow_pickle=False) as a:
            old[run] = {k: a[k].copy() for k in a.files}
        a = old[run]
        assert list(a["targets"]) == TARGETS
        assert np.array_equal(a["file_names"], valid.file_name.to_numpy()), "기존 검증 순서와 다릅니다."
        assert np.array_equal(a["y_true"], valid[TARGETS].to_numpy()), "기존 정답과 다릅니다."
        assert a["probs"].shape == (5792, 9) and np.isfinite(a["probs"]).all()
        assert ((a["probs"] >= 0) & (a["probs"] <= 1)).all()
print("기존 5개 결과와 검증 행 순서·정답 일치 확인 완료")


## 3. 우선순위 1 — 증상별 오류 분석
오심의 낮은 점수가 precision 문제인지 recall 문제인지 확인합니다.
표본을 읽고 근거 누락·부정문·질문·간접 표현·잘린 문맥을 구분하세요.
오심과 오한을 혼동한 규칙을 만들거나 정답을 임의 수정하지 않습니다.
아래 CSV에는 통화 원문이 있으므로 GitHub에 올리지 마세요.


In [ ]:
def label_report(y, probs):
    pred = probs >= .5
    p, r, f, support = precision_recall_fscore_support(y, pred, average=None, zero_division=0)
    return pd.DataFrame({"symptom": TARGETS, "precision": p, "recall": r, "f1": f,
                         "support": support, "predicted_positive": pred.sum(axis=0)})
for name, a in old.items():
    print(name, "Macro F1:", f1_score(a["y_true"], a["probs"] >= .5, average="macro", zero_division=0))
display(label_report(old["e0"]["y_true"], old["e0"]["probs"]).sort_values("f1"))
j = TARGETS.index("오심")
a = old["e0"]
err = valid[["file_name","text"]].copy()
err["truth"] = a["y_true"][:,j]
err["probability"] = a["probs"][:,j]
err["prediction"] = (err.probability >= .5).astype(int)
err = err[err.truth != err.prediction].copy()
err["error_type"] = np.where(err.truth == 1, "FN: 오심 놓침", "FP: 오심 과예측")
err.to_csv(OUT / "e0_nausea_errors_private.csv", index=False, encoding="utf-8-sig")
for kind, group in err.groupby("error_type"):
    print(kind, len(group))
    display(group.sample(min(10, len(group)), random_state=42))


## 4. 동일한 First-512 입력 준비
세 head 모두 동일한 토큰·학습 설정을 씁니다. Mean/Attention은 패딩과 CLS/SEP를 제외한 본문 표현을 모읍니다.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
class Calls(Dataset):
    def __init__(self, frame):
        self.items = tokenizer(frame.text.tolist(), truncation=True, max_length=512,
                               padding=False, return_token_type_ids=False)
        self.labels = frame[TARGETS].to_numpy(dtype=np.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {**{k: v[i] for k,v in self.items.items()}, "labels": self.labels[i]}
train_ds, valid_ds = Calls(train), Calls(valid)
def collate(batch):
    x = tokenizer.pad([{k:v for k,v in b.items() if k != "labels"} for b in batch],
                      return_tensors="pt")
    x["labels"] = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float32)
    return x


## 5. 세 head 정의
CLS: 첫 토큰 → Dense/Tanh → 9개 출력.
Mean: 본문 평균 → 같은 Dense/Tanh → 9개 출력.
Label attention: 증상마다 본문 토큰 가중합 → 공유 Dense/Tanh → 증상별 출력.
출력 sigmoid ≥ 0.5. 학습 손실은 모두 같은 BCE입니다.
가중치 저장 시 head_type도 config에 남기며, 복원에는 아래 클래스 정의가 필요합니다.


In [ ]:
class SymptomModel(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        h = config.hidden_size
        self.head_type = getattr(config, "head_type", "cls")
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.dense = nn.Linear(h, h)
        self.out_proj = nn.Linear(h, config.num_labels)
        if self.head_type == "label_attention":
            self.attention = nn.Linear(h, config.num_labels, bias=False)
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        hidden = self.roberta(input_ids=input_ids, attention_mask=attention_mask,
                              return_dict=True).last_hidden_state
        mask = attention_mask.bool().clone()
        mask[:,0] = False
        last = attention_mask.sum(1).long() - 1
        mask[torch.arange(len(mask), device=mask.device), last] = False
        empty = ~mask.any(dim=1)
        mask[empty,0] = True
        if self.head_type == "cls":
            pooled = hidden[:,0]
        elif self.head_type == "mean":
            pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        else:
            scores = self.attention(hidden).float().masked_fill(~mask.unsqueeze(-1), -1e9)
            weights = scores.softmax(dim=1).to(hidden.dtype)
            pooled = torch.einsum("blc,blh->bch", weights, hidden)
        x = self.dropout(torch.tanh(self.dense(self.dropout(pooled))))
        if self.head_type == "label_attention":
            logits = (x * self.out_proj.weight.unsqueeze(0)).sum(-1) + self.out_proj.bias
        else:
            logits = self.out_proj(x)
        loss = None if labels is None else nn.functional.binary_cross_entropy_with_logits(
            logits.float(), labels.float())
        return SequenceClassifierOutput(loss=loss, logits=logits)

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    return {"macro_f1": f1_score(p.label_ids, logits >= 0, average="macro", zero_division=0)}

# 다운로드·학습 없는 구조 검사
tiny = RobertaConfig(vocab_size=32, hidden_size=24, num_hidden_layers=1,
                     num_attention_heads=4, intermediate_size=48, num_labels=9)
for kind in ["cls","mean","label_attention"]:
    tiny.head_type = kind
    m = SymptomModel(tiny).eval()
    with torch.no_grad():
        v = m(torch.tensor([[0,5,6,2,1],[0,2,1,1,1]]),
              torch.tensor([[1,1,1,1,0],[1,1,0,0,0]]), torch.zeros(2,9))
    assert v.logits.shape == (2,9) and torch.isfinite(v.loss)
del m
print("head 구조 검사 OK")


## 6. 학습 함수
각 실험은 사전학습 모델에서 독립적으로 시작합니다. 기존 결과를 덮어쓰지 않습니다.
마지막 epoch 결과를 주 비교에 사용합니다(과거 E0와 같은 방침).
중단 후 같은 실험을 계속하면 남아 있는 checkpoint에서 복원합니다.
모델 저장과 작은 결과 백업은 별개입니다. /content는 임시 공간입니다.


In [ ]:
def run_experiment(kind):
    assert kind in ["cls","mean","label_attention"]
    out = OUT / kind
    if (out / "metrics.json").exists():
        print("완료된 실험입니다. 저장 결과:", json.loads((out / "metrics.json").read_text()))
        return
    out.mkdir(exist_ok=True)
    checkpoints = sorted((out / "checkpoints").glob("checkpoint-*"),
                         key=lambda p: int(p.name.split("-")[-1]))
    resume = str(checkpoints[-1]) if checkpoints else None
    set_seed(42)
    config = RobertaConfig.from_pretrained(MODEL_NAME)
    config.num_labels = 9
    config.id2label = dict(enumerate(TARGETS))
    config.label2id = {t:i for i,t in enumerate(TARGETS)}
    config.head_type = kind
    model = SymptomModel.from_pretrained(MODEL_NAME, config=config)
    kwargs = dict(output_dir=str(out / "checkpoints"), num_train_epochs=4,
                  per_device_train_batch_size=8, per_device_eval_batch_size=16,
                  gradient_accumulation_steps=1, learning_rate=2e-5, warmup_steps=1000,
                  weight_decay=.01, logging_steps=200, fp16=True, report_to="none",
                  seed=42, data_seed=42, save_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=False)
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
    kwargs[key] = "epoch"
    provenance = {"head": kind, "model": MODEL_NAME, "model_revision": getattr(config,"_commit_hash",None),
                  "threshold": .5, "loss": "BCE", "selection": "final_epoch",
                  "input": "first_512", "targets": TARGETS, "environment": ENV,
                  "manifest_sha256": hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
                  "training": kwargs}
    previous = out / "run_config.json"
    if previous.exists():
        assert json.loads(previous.read_text()) == provenance, "기존 설정과 다릅니다. OUT 경로를 변경하세요."
    previous.write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
    trainer = Trainer(model=model, args=TrainingArguments(**kwargs), train_dataset=train_ds,
                      eval_dataset=valid_ds, data_collator=collate, compute_metrics=compute_metrics)
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(out / "model"))
    tokenizer.save_pretrained(out / "model")
    logits = trainer.predict(valid_ds).predictions
    if isinstance(logits, tuple): logits = logits[0]
    probs = torch.sigmoid(torch.as_tensor(logits).float()).numpy()
    y = valid[TARGETS].to_numpy(dtype=int)
    np.savez_compressed(out / "dev_predictions.npz", probs=probs, y_true=y,
                        file_names=valid.file_name.to_numpy(dtype=str), targets=np.asarray(TARGETS))
    report = label_report(y, probs)
    report.to_csv(out / "per_label.csv", index=False)
    result = {"fixed_threshold": .5,
              "fixed_threshold_macro_f1": float(f1_score(y, probs >= .5, average="macro", zero_division=0)),
              "head": kind, "evaluation": "internal validation", "selection": "final_epoch"}
    (out / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))
    (out / "metrics.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
    print(result)
    display(report.sort_values("f1"))
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()


## 7. CLS 대조군
동일한 새 코드에서 비교 기준을 만듭니다. 기존 E0 수치도 별도로 보존합니다.


In [ ]:
run_experiment("cls")


## 8. 우선순위 2 — Mean pooling
앞 셀의 학습이 끝난 뒤 실행하세요.


In [ ]:
run_experiment("mean")


## 9. 우선순위 3 — 증상별 Attention
Mean 결과에 관계없이 사전에 계획한 비교입니다.


In [ ]:
run_experiment("label_attention")


## 10. 결과 모으기
세 실험의 차이가 작다면 seed를 추가해 반복해야 합니다. 최고 점수 하나로 일반화 성능 향상을 확정하지 않습니다.


In [ ]:
summary = [{"experiment": "historical_" + name,
            "macro_f1": f1_score(a["y_true"], a["probs"] >= .5, average="macro", zero_division=0)}
           for name,a in old.items()]
for kind in ["cls","mean","label_attention"]:
    p = OUT / kind / "metrics.json"
    if p.exists():
        r = json.loads(p.read_text())
        summary.append({"experiment": kind, "macro_f1": r["fixed_threshold_macro_f1"]})
summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT / "summary.csv", index=False)


## 11. 매 실험 후 실행 가능한 작은 결과 백업
예측·지표·분할·로그·실험 설정을 포함합니다. 학습 가중치와 원문 오류 CSV는 제외합니다. 노트북 코드는 별도로 저장하세요.
이 백업으로 내부 성능 재계산은 가능하지만 새 데이터 추론·학습 재개에는 모델/checkpoint 백업도 필요합니다.
노트북 자체도 파일 → 다운로드 → .ipynb로 저장하세요.


In [ ]:
archive_path = Path("/content/m3_head_results.tar.gz")
with tarfile.open(archive_path, "w:gz") as a:
    for p in OUT.rglob("*"):
        if p.is_file() and "model" not in p.relative_to(OUT).parts and "checkpoints" not in p.relative_to(OUT).parts:
            if p.name != "e0_nausea_errors_private.csv":
                a.add(p, arcname=str(p.relative_to(OUT)))
print("SHA256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())
files.download(str(archive_path))


## 12. 선택: 모델까지 Drive 백업
Drive 여유 공간을 먼저 확인하세요. 모델은 크며 작은 결과 백업에는 없습니다.
아래 BACKUP_MODELS를 True로 바꿔야 복사합니다. checkpoint를 포함하므로 용량이 수 GB 이상 필요할 수 있습니다.
새 경로만 허용하고 기존 백업은 덮어쓰지 않습니다.


In [ ]:
BACKUP_MODELS = False
if BACKUP_MODELS:
    import shutil
    from datetime import datetime
    destination = Path("/content/drive/MyDrive/DDC_M3/backups") / ("heads_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(OUT, destination)
    print("모델·체크포인트 포함 백업:", destination)
else:
    print("모델 백업을 건너뜁니다. 결과 압축파일에는 모델이 없습니다.")


복원 예시 (모델 클래스 정의 셀을 먼저 실행):
`SymptomModel.from_pretrained("/content/.../model")`

설계 참고: [Hugging Face Trainer](https://huggingface.co/docs/transformers/main_classes/trainer),
[Label-Specific Attention 연구](https://aclanthology.org/D19-1044/).
이 노트북은 해당 논문 전체 재현이 아닌 간단한 증상별 attention 비교 구현입니다.
